# FIFO 5秒观测：启动摘要复用候选

保留原版。仅减少启动前重复SHA计算：每次校验仍读真实文件，只复用内容完全相同的摘要；在原STARTUP阶段前清空缓存。180秒总启动保护、700点预热、C-E 80ms、原算法/电路及退出策略不变。正式最多5秒；预热另计。结果保留、不自动重试。

先结束其他硬件任务，接好摄像头与HDMI。第一单元无硬件；第三单元由你确认后才加载原电路和采集。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
from IPython.display import display, FileLink
assert sys.platform == 'linux', '仅在板子运行'
root = Path.cwd()
assert root.name == 'v3b25 fifo startup5', '请在新的启动修复目录运行'
from startup_digest_guard import verify
checked = verify(root)
print({k: checked[k] for k in ('bundle_validated', 'source_members_unchanged', 'package_members', 'manifest_sha256')})
from run_startup_digest5 import short, prepare, run
short_report = short(expected_manifest_sha256=checked['manifest_sha256'])
if (root/'lean-ps-short-results.zip').is_file():
    display(FileLink('lean-ps-short-results.zip'))
assert short_report['passed'], '原无硬件短退出未通过，保留结果，不启动硬件'


In [ ]:
# 只准备浏览器显示；等绿色就绪，再运行下一单元。
prepared = prepare(expected_manifest_sha256=checked['manifest_sha256'])


In [ ]:
assert not globals().get('_startup_digest_used', False), '已尝试，保留结果，不重跑'
confirmation = input('其他任务已结束、放稳手指；允许加载原电路并做一次最多5秒正式观测？输入 RUN-FIFO-STARTUP5: ')
assert confirmation == 'RUN-FIFO-STARTUP5', '未确认，不启动硬件'
_startup_digest_used = True
print('启动总保护仍为180秒；预热另计，正式最多5秒。请保留输出，不中断、不重跑。', flush=True)
try:
    report = run(hardware=True, confirmation=confirmation,
                 expected_manifest_sha256=checked['manifest_sha256'])
finally:
    if (root/'startup-digest5-results.zip').is_file():
        display(FileLink('startup-digest5-results.zip'))
    else:
        print('停止/导出尚未确认，保留当前输出和目录，不关闭仍在收尾的内核。')
